### MIDDLEWARE
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ['LLAMA_API_KEY'] = os.getenv("LLAMA_API_KEY")

Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.
Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


agent = create_agent(
    model=init_chat_model("groq:openai/gpt-oss-20b"),
    middleware=[SummarizationMiddleware(
        model=init_chat_model("groq:openai/gpt-oss-20b"),
        trigger=("messages",10),
        keep=("messages", 4)
    )],
    checkpointer=InMemorySaver()
)

In [8]:
config = {"configurable": {"thread_id":"test_1"} }

questions =  [
    "What is 2+2",
    "What is 5*10",
    "What is 100/10",
    "What is 3*3",
    "What is 4*4"
]

for q in questions:
    result = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f'Messages: { result}')
    print(f"Message length: {len(result['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='b5ce7037-bedf-4120-9158-f4ea77c6c6fa'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks: "What is 2+2". Simple. Provide answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 77, 'total_tokens': 116, 'completion_time': 0.040186063, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.00428428, 'prompt_tokens_details': None, 'queue_time': 0.156508509, 'total_time': 0.044470343}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1d4ae-1114-73a4-bd12-4967c392c33c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 39, 'total_tokens': 116, 'output_token_details': {'reasoning': 20}}), HumanMessage(

In [21]:
## Tools
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

@tool
def search_hotels(city: str):
    """Search hotels - return long response of hotels. """
    return f""" Hotels in {city}:
    1. Grand Hotel - 5 stars, $350/night, spa and pool
    2. City Inn - 3 stars, $180/night, buisiness center
    3. Budget Stay - 2 stars, $90/night, free wifi
    """


agent = create_agent(
    model=init_chat_model("groq:openai/gpt-oss-120b"),
    tools=[search_hotels],
    middleware=[SummarizationMiddleware(
        model=init_chat_model("groq:qwen/qwen3.8-27b"),
        trigger=("tokens",580),
        keep=("tokens", 200)
    )],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id":"test_2"} }

def count_tokens(messages):
    total_chars =  sum(len(m.content) for m in messages)
    return total_chars // 4

In [22]:
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]

for city in cities:
    result = agent.invoke({"messages": [HumanMessage(content=f"Find hotels in {city}")]}, config=config)
    total_tokens = count_tokens(result['messages'])
    print(f'Messages: { result["messages"]}')
    print(f"city: {city}, Message length: {len(result['messages'])}, Total tokens: {total_tokens}")

Messages: [HumanMessage(content='Find hotels in New York', additional_kwargs={}, response_metadata={}, id='9fd4e3a5-a020-48df-a25c-940d0d78e547'), AIMessage(content='', additional_kwargs={'reasoning_content': "User wants to find hotels in New York. We have a function search_hotels that takes city. We'll call it.", 'tool_calls': [{'id': 'fc_ef5dd43d-2baa-4fdf-987d-271fecb3c935', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 128, 'total_tokens': 182, 'completion_time': 0.114360237, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.026356635, 'prompt_tokens_details': None, 'queue_time': 0.375290073, 'total_time': 0.140716872}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1d4ed-3805-770a-b845-

Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).
Compliance workflows where human oversight is mandatory.
Long-running conversations where human feedback guides the agent.

In [24]:
## Human in the loop
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


def read_email_tool(email_id: str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID {email_id}"


def send_email_tool(recipient :str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'."


In [25]:


agent = create_agent(
    model=init_chat_model("groq:openai/gpt-oss-120b"),
    tools=[read_email_tool, send_email_tool],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool": {
                "allowed_decisions": ["approve","edit","reject"]
            },
            "read_email_tool": False

        }
    )],
    checkpointer=InMemorySaver()
)

In [26]:
config = {"configurable": {"thread_id":"approve_test"} }

result = agent.invoke({"messages": [HumanMessage(content=f"Send an email to john@example.com with subject 'Hello' and body 'How are you?'")]}, config=config)

result

{'messages': [HumanMessage(content="Send an email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='22578b42-617e-478a-a39c-910abf509ef0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide recipient, subject, body.', 'tool_calls': [{'id': 'fc_b667a04c-faff-4ca1-817c-b4f396efc910', 'function': {'arguments': '{"body":"How are you?","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 175, 'total_tokens': 239, 'completion_time': 0.134865359, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.007290047, 'prompt_tokens_details': None, 'queue_time': 0.28729571, 'total_time': 0.142155406}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_

In [ ]:
#Step 2 Approve

from langgraph.types import Command



if "__interrupt__" in result:
    print(" Pause! approving ")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    { "type": "approve" }
                ]
            }
        ), config=config
    )

print(f" Result after approval: {result['messages'][-1].content}")


 Pause! approving 
 Result after approval: The email has been sent to **john@example.com** with the subject **“Hello”** and the body **“How are you?”**. Let me know if there’s anything else you’d like to do!


In [29]:
result

{'messages': [HumanMessage(content="Send an email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='22578b42-617e-478a-a39c-910abf509ef0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide recipient, subject, body.', 'tool_calls': [{'id': 'fc_b667a04c-faff-4ca1-817c-b4f396efc910', 'function': {'arguments': '{"body":"How are you?","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 175, 'total_tokens': 239, 'completion_time': 0.134865359, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.007290047, 'prompt_tokens_details': None, 'queue_time': 0.28729571, 'total_time': 0.142155406}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_

In [32]:
#### step 2 Edit and approve


if "__interrupt__" in result:
    print(" Pause! editing ")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    { "type": "edit", 
                     "edited_action": {
                         "name":  "send_email_tool",
                            "args": {
                                "recipient": "correct@mail.com",
                                "subject": "Corrected Subject",
                                "body": " this is edited by Human before sending."
                            }
                        }
                     }
                ]
            }
        ), config=config
    )


    result


